# 2/2 — Evaluate, ablate and export Qwen3-8B (Kaggle, GPU T4 x2)

**Before running:** *Add Input → Your Work →* the training notebook (its version output contains `qwen3-8b-lora`).
Every stage writes `results/eval_<tag>.json`; a stage whose file already exists (also from an attached previous
version of *this* notebook) is skipped, so the notebook can be committed several times until everything is done.

**Fair comparison:** `--limit N` always takes the first N rows of `test.jsonl`. Base and fine-tuned main runs use
`EVAL_LIMIT`; the ablation uses the first `ABL_LIMIT` rows (a subset of the same rows), and the E-level ablation
numbers are computed from the main runs restricted to those rows.

## 1. Code and dependencies

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/yarenerisen/bist100-agentic-financial-harness.git repo
%cd /kaggle/working/repo/bist_llm
!pip install -q -U "transformers>=4.45" "peft>=0.13" "bitsandbytes>=0.44" accelerate pyarrow liger-kernel
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Settings and inputs

In [ ]:
import glob, os, shutil, json
MODEL       = 'Qwen/Qwen3-8B'
RESULTS     = '/kaggle/working/results'
EVAL_LIMIT  = 40     # same as the training notebook
ABL_LIMIT   = 20     # ablation subset = first ABL_LIMIT rows of the same test file
REPRO_LIMIT = 10     # reproducibility subset (3 repeats each)
RUN_MAIN, RUN_REPRO, RUN_ABLATION, RUN_GGUF = True, True, True, False

cands = glob.glob('/kaggle/input/*/qwen3-8b-lora') + glob.glob('/kaggle/input/*/*/qwen3-8b-lora')
ADAPTER = cands[0] if cands else None
print('adapter:', ADAPTER)
assert ADAPTER and os.path.exists(f'{ADAPTER}/adapter_config.json'), 'attach the training notebook output as input'
os.makedirs(RESULTS, exist_ok=True)
for f in glob.glob('/kaggle/input/*/results/eval_*.json') + glob.glob('/kaggle/input/*/*/results/eval_*.json'):
    if not os.path.exists(os.path.join(RESULTS, os.path.basename(f))):
        shutil.copy(f, RESULTS); print('reused', os.path.basename(f))

def run_eval(tag, model, mode='E', limit=EVAL_LIMIT, repeats=1):
    if os.path.exists(f'{RESULTS}/eval_{tag}.json'):
        print('skip (exists):', tag); return
    !python -m harness.evaluate --backend hf --model {model} --load-4bit --mode {mode} --limit {limit} --repeats {repeats} --tag {tag} --out-dir {RESULTS}

## 3. Main comparison: base vs fine-tuned, full harness, identical rows

In [ ]:
if RUN_MAIN:
    run_eval('base_E', MODEL)
    run_eval('finetuned_E', ADAPTER)

## 4. Reproducibility: same question 3 times

In [ ]:
if RUN_REPRO:
    run_eval('finetuned_repro', ADAPTER, limit=REPRO_LIMIT, repeats=3)

## 5. Harness ablation (same rows for every level)
A = model gets the reference tool outputs as text, no tools/checks · B = tools, no validation · C = stateful harness ·
D = + evidence and quality gate · E = + memory and human review (taken from the main runs, first ABL_LIMIT rows).
The raw baseline *without any data* (`training/baseline/base_model_raw_outputs.jsonl`) is scored separately by
`harness/score_raw.py` and reported as its own row.

In [ ]:
if RUN_ABLATION:
    for m in 'ABCD':
        run_eval(f'finetuned_{m}', ADAPTER, mode=m, limit=ABL_LIMIT)
    for m in 'AB':
        run_eval(f'base_{m}', MODEL, mode=m, limit=ABL_LIMIT)

## 6. Results tables (main runs and ablation on identical rows)

In [ ]:
import pandas as pd, sys
sys.path.insert(0, '.')
from harness.evaluate import summarize
load = lambda t: json.load(open(f'{RESULTS}/eval_{t}.json')) if os.path.exists(f'{RESULTS}/eval_{t}.json') else None
cols = ['n', 'numerical_hallucination_rate_pct', 'invalid_tool_call_rate_pct', 'correct_tool_sequence_pct', 'correct_tool_args_pct',
        'unsupported_conclusion_rate_pct', 'raw_answer_rule_violation_rate_pct', 'final_answer_valid_pct', 'evidence_completeness_pct',
        'reproducibility_pct', 'mean_tool_calls', 'mean_latency_s']
main = {t: load(t)['overall'] for t in ('base_E', 'finetuned_E') if load(t)}
display(pd.DataFrame(main).T.reindex(columns=cols))
for t in ('base_E', 'finetuned_E'):
    if load(t): print(t, 'by fundamentals source:', {k: (v['n'], v['numerical_hallucination_rate_pct']) for k, v in load(t)['by_fundamentals_source'].items()})
abl = {}
for who in ('base', 'finetuned'):
    for m in 'ABCD':
        if load(f'{who}_{m}'): abl[f'{who}_{m}'] = load(f'{who}_{m}')['overall']
    if load(f'{who}_E'): abl[f'{who}_E'] = summarize(load(f'{who}_E')['rows'][:ABL_LIMIT])
raw = json.load(open('training/baseline/base_model_raw_scores.json'))['overall'] if os.path.exists('training/baseline/base_model_raw_scores.json') else None
display(pd.DataFrame(abl).T.reindex(columns=cols))
if raw: print('raw base model without data (5 instructor-style questions):', raw)
if load('finetuned_repro'): print('reproducibility:', load('finetuned_repro')['overall']['reproducibility_pct'], '%')

## 7. (optional) GGUF Q4_K_M for local offline inference (~5 GB, fits an 8 GB GPU)

In [ ]:
if RUN_GGUF:
    !python training/finetune_qlora.py --merge --model {MODEL} --out {ADAPTER} --merged-out /tmp/merged
    !git clone -q https://github.com/ggerganov/llama.cpp /tmp/llama.cpp && pip install -q -r /tmp/llama.cpp/requirements.txt
    !python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outfile /tmp/bist-f16.gguf --outtype f16
    !cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null
    !/tmp/llama.cpp/build/bin/llama-quantize /tmp/bist-f16.gguf /kaggle/working/bist-analyst-qwen3-8b-Q4_K_M.gguf Q4_K_M
    !ls -lh /kaggle/working/*.gguf